Agents

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline, HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.tools import StructuredTool, BaseTool
from pydantic import BaseModel, Field
from typing import Type
from langchain_classic.agents import AgentExecutor, create_react_agent # This is older implementation of agent, newer one is in langchain_core.agents
from langchain_classic import hub
from dotenv import load_dotenv

In [7]:
load_dotenv()
hf_llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3.1-8B-Instruct",
    task="conversational",
    max_new_tokens=120,
    temperature=0.7,
)
llm = ChatHuggingFace(llm=hf_llm)

In [8]:
prompt = hub.pull("hwchase17/react")

In [9]:
agent = create_react_agent(
    llm=llm,
    tools=[DuckDuckGoSearchRun()],
    prompt=prompt
)

In [15]:
print(agent)
print(agent.input_schema)
print(agent.output_schema)

first=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
}) middle=[PromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={}, partial_variables={'tools': 'duckduckgo_search - A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.', 'tool_names': 'duckduckgo_search'}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/O

In [18]:
agent_executor = AgentExecutor(
    agent=agent, 
    tools=[DuckDuckGoSearchRun()],
    verbose=True
)

In [19]:
print(agent_executor)

verbose=True agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={}, partial_variables={'tools': 'duckduckgo_search - A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.', 'tool_names': 'duckduckgo_search'}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this

In [ ]:
# Execute the agent with a query
agent_executor.invoke({
    "input": "What is the temperature in Bengaluru today?"
})



> Entering new AgentExecutor chain...
Thought: The temperature in Bengaluru can vary greatly depending on the time of year and current weather conditions. To find the current temperature, I should use a search engine to look up current weather information for Bengaluru.
Action: duckduckgo_search
Action Input: "current temperature in Bengaluru today"
ObservCheckcurrentconditionsinBengaluru, Karnataka, India with radar, hourly, and more. CurrentweatherinBengaluruand forecast fortoday, tomorrow, and next 14 days. MeteoTrend: WeatherinBengalurufortoday, tomorrow and week.Temperatureand humidity of air, pressure, speed and wind direction, precipitation, sunrise, sunset, moon rise, moon set. Bengaluru, India -Currenttemperatureand weather conditions. Detailed hourly weather forecast fortoday- including weather conditions,temperature, pressure, humidity, precipitation, dewpoint, wind, visibility, and UV index data. Get the latestBengaluruweather updates! Checktoday’s forecast,temperature, h

{'input': 'What is the temperature in Bengaluru today?',
 'output': 'The current temperature in Bengaluru is 26.1°C with Clear conditions.'}